# 01 - Exploratory Data Analysis

Structural overview of `data/application_train.csv`: shape and data types.

Reusable logic lives in `src/eda.py` (pure, tested functions); this notebook only loads data and calls them.

## 1. Setup

In [1]:
import sys
from pathlib import Path

import pandas as pd

# Add project root to sys.path so `src` can be imported from the notebooks/ folder
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.eda import (
    classify_variables,
    imbalance_ratio,
    missing_summary,
    suggest_missing_strategy,
    summarize_structure,
    target_distribution,
)

## 2. Load data

In [2]:
# Read the raw training set (~166 MB, may take a few seconds)
DATA_PATH = PROJECT_ROOT / "data" / "application_train.csv"
df = pd.read_csv(DATA_PATH)
df.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


## 3. Structural summary

In [3]:
# Compute shape and dtype information
summary = summarize_structure(df)

print(f"Rows: {summary['n_rows']:,}")
print(f"Columns: {summary['n_cols']}")

Rows: 307,511
Columns: 122


In [4]:
# Number of columns per data type
pd.Series(summary["dtype_counts"], name="n_columns")

float64    65
int64      41
str        16
Name: n_columns, dtype: int64

In [5]:
# Data type of every column, as a table for easy scrolling/filtering
pd.Series(summary["dtypes"], name="dtype").to_frame()

,dtype
SK_ID_CURR,int64
TARGET,int64
NAME_CONTRACT_TYPE,str
CODE_GENDER,str
FLAG_OWN_CAR,str
...,...
AMT_REQ_CREDIT_BUREAU_DAY,float64
AMT_REQ_CREDIT_BUREAU_WEEK,float64
AMT_REQ_CREDIT_BUREAU_MON,float64
AMT_REQ_CREDIT_BUREAU_QRT,float64


## 4. Default rate (target distribution)

`TARGET = 0`: client repaid the loan; `TARGET = 1`: client defaulted (payment difficulties).

A strong imbalance affects model choice, evaluation metrics (prefer ROC-AUC / Gini / KS over accuracy) and the need for stratified train/test splits.

In [6]:
# Count and share of each class
dist = target_distribution(df)
dist

,count,ratio
TARGET,,
0,282686,0.919271
1,24825,0.080729


In [7]:
# Headline numbers: default rate and class imbalance
print(f"Default rate: {dist.loc[1, 'ratio']:.2%}")
print(f"Imbalance: {imbalance_ratio(dist):.1f} repaid clients per defaulted client")

Default rate: 8.07%
Imbalance: 11.4 repaid clients per defaulted client


## 5. Missing values

Treatment guideline for a WoE-based scorecard (thresholds are adjustable):

| `pct_missing` | Strategy | Rationale |
|---|---|---|
| ≥ 60% | `drop_candidate` | Too sparse to be reliable; keep only if IV shows strong signal |
| 5% – 60% | `missing_bin` | Missingness itself may be predictive; treat it as a separate WoE bin |
| < 5% | `impute` | Median (numeric) / mode (categorical) has negligible impact |

In [8]:
# Columns with missing values, sorted by missing percentage
missing = missing_summary(df)
missing["strategy"] = missing["pct_missing"].apply(suggest_missing_strategy)

print(f"{len(missing)} / {df.shape[1]} columns have missing values")
missing.round(2)

67 / 122 columns have missing values


,n_missing,pct_missing,dtype,strategy
COMMONAREA_MEDI,214865,69.87,float64,drop_candidate
COMMONAREA_MODE,214865,69.87,float64,drop_candidate
COMMONAREA_AVG,214865,69.87,float64,drop_candidate
NONLIVINGAPARTMENTS_MODE,213514,69.43,float64,drop_candidate
NONLIVINGAPARTMENTS_MEDI,213514,69.43,float64,drop_candidate
...,...,...,...,...
EXT_SOURCE_2,660,0.21,float64,impute
AMT_GOODS_PRICE,278,0.09,float64,impute
AMT_ANNUITY,12,0.00,float64,impute
CNT_FAM_MEMBERS,2,0.00,float64,impute


In [9]:
# Number of columns per suggested strategy
missing["strategy"].value_counts()

strategy
missing_bin       40
drop_candidate    17
impute            10
Name: count, dtype: int64

In [10]:
# Missing-rate overview by dtype: are numeric or categorical columns more affected?
missing.groupby("dtype")["pct_missing"].agg(["count", "mean", "max"]).round(2)

,count,mean,max
dtype,,,
float64,61,44.72,69.87
str,6,41.43,68.39


## 6. Variable types

Binning for WoE/IV depends on variable type:

- **Numeric**: split into intervals (e.g. quantile / optimal binning), then compute WoE per interval.
- **Categorical**: each category is a bin; rare categories may be merged.

`dtype` alone is not enough: binary `FLAG_*` columns and low-cardinality integers (e.g. `REGION_RATING_CLIENT` = 1/2/3) are stored as numbers but behave as categories. ID and target columns are excluded.

In [11]:
# Count variables with a sparse tail: bin as numeric (e.g. 0 | 1 | >=2)
# instead of one unstable WoE bin per distinct value
COUNT_VARS = (
    "AMT_REQ_CREDIT_BUREAU_HOUR",
    "AMT_REQ_CREDIT_BUREAU_DAY",
    "AMT_REQ_CREDIT_BUREAU_WEEK",
    "DEF_30_CNT_SOCIAL_CIRCLE",
    "DEF_60_CNT_SOCIAL_CIRCLE",
)

# Classify features (numeric columns with <= 10 distinct values -> categorical,
# except the forced count variables above)
var_types = classify_variables(df, max_categories=10, force_numeric=COUNT_VARS)
var_types["var_type"].value_counts()

var_type
numeric        70
categorical    50
Name: count, dtype: int64

In [12]:
# Cross-check: how each storage dtype was classified
pd.crosstab(var_types["dtype"], var_types["var_type"])

var_type,categorical,numeric
dtype,,
float64,0,65
int64,34,5
str,16,0


In [13]:
# Feature lists to be reused in the WoE/IV phase
numeric_cols = var_types.index[var_types["var_type"] == "numeric"].tolist()
categorical_cols = var_types.index[var_types["var_type"] == "categorical"].tolist()

print(f"Numeric ({len(numeric_cols)}):", numeric_cols)
print(f"\nCategorical ({len(categorical_cols)}):", categorical_cols)

Numeric (70): ['AMT_REQ_CREDIT_BUREAU_HOUR', 'DEF_60_CNT_SOCIAL_CIRCLE', 'AMT_REQ_CREDIT_BUREAU_DAY', 'AMT_REQ_CREDIT_BUREAU_WEEK', 'DEF_30_CNT_SOCIAL_CIRCLE', 'AMT_REQ_CREDIT_BUREAU_QRT', 'CNT_CHILDREN', 'CNT_FAM_MEMBERS', 'HOUR_APPR_PROCESS_START', 'AMT_REQ_CREDIT_BUREAU_MON', 'FLOORSMAX_MODE', 'FLOORSMIN_MODE', 'AMT_REQ_CREDIT_BUREAU_YEAR', 'ELEVATORS_MODE', 'ENTRANCES_MODE', 'OBS_30_CNT_SOCIAL_CIRCLE', 'OBS_60_CNT_SOCIAL_CIRCLE', 'ELEVATORS_MEDI', 'ENTRANCES_MEDI', 'FLOORSMIN_MEDI', 'FLOORSMAX_MEDI', 'OWN_CAR_AGE', 'REGION_POPULATION_RELATIVE', 'YEARS_BUILD_AVG', 'YEARS_BUILD_MEDI', 'YEARS_BUILD_MODE', 'NONLIVINGAPARTMENTS_MODE', 'NONLIVINGAPARTMENTS_MEDI', 'YEARS_BEGINEXPLUATATION_MODE', 'YEARS_BEGINEXPLUATATION_MEDI', 'ELEVATORS_AVG', 'YEARS_BEGINEXPLUATATION_AVG', 'ENTRANCES_AVG', 'FLOORSMIN_AVG', 'NONLIVINGAPARTMENTS_AVG', 'FLOORSMAX_AVG', 'LIVINGAPARTMENTS_MODE', 'APARTMENTS_MODE', 'EXT_SOURCE_3', 'AMT_GOODS_PRICE', 'LIVINGAPARTMENTS_MEDI', 'APARTMENTS_MEDI', 'LIVINGAPARTMENTS

In [14]:
# Borderline cases: numeric dtype classified as categorical -> review manually
var_types[(var_types["var_type"] == "categorical") & (var_types["dtype"] != "str")]

,dtype,n_unique,var_type
FLAG_MOBIL,int64,2,categorical
FLAG_EMP_PHONE,int64,2,categorical
FLAG_WORK_PHONE,int64,2,categorical
FLAG_CONT_MOBILE,int64,2,categorical
FLAG_PHONE,int64,2,categorical
FLAG_EMAIL,int64,2,categorical
REG_REGION_NOT_LIVE_REGION,int64,2,categorical
REG_REGION_NOT_WORK_REGION,int64,2,categorical
LIVE_REGION_NOT_WORK_REGION,int64,2,categorical
REG_CITY_NOT_LIVE_CITY,int64,2,categorical
